In [1]:
import os
import pickle
import re
from opencc import OpenCC
from seqeval.metrics import classification_report

t2s = OpenCC('t2s')
s2t = OpenCC('s2t')
remove_list = ["", " "]

def generate_answer(context, ent):
    answer = ['O'] * len(context)
    if "無" in ent:
        return answer

    converters = [
        lambda x: x,
        lambda x: x.lower(),
        lambda x: x.upper(),
        lambda x: t2s.convert(x),
        lambda x: s2t.convert(x),
    ]

    for e in ent:
        found = False
        for conv in converters:
            ctx_conv = conv(context)
            ent_conv = conv(e)
            start_idx = ctx_conv.find(ent_conv)
            if start_idx != -1:
                answer[start_idx] = 'B'
                for i in range(1, len(ent_conv)):
                    answer[start_idx + i] = 'I'
                found = True
                break

        if not found:
            print(f"找不到實體 '{e}' 在上下文中 '{context}'")

    return answer

def process_data(data_list):

    result_data = []
    for data in data_list:
        last_content = data[-1]['content']
        
        p_name = data[0]['context']

        ent = last_content.split("、")
        ent = [e for e in ent if e not in remove_list]  # 移除空字串
        ent = [s2t.convert(e) for e in ent]  # 簡繁轉換

        answer = generate_answer(p_name, ent)

        result_data.append({
            'context': p_name,
            'question': '品牌',
            'answer': answer
        })
    return result_data

directory_path = "./results_test_1500_brand_qwen_3_8B_bf16"
# directory_path = "./results_test_1500_brand_qwen_3_8B_bf16_new_prompt"

data_list = []

for filename in os.listdir(directory_path):
    if "品牌" in filename and filename.endswith(".pkl"):
        file_path = os.path.join(directory_path, filename)
        
        try:
            with open(file_path, "rb") as file:
                data = pickle.load(file)
                data_list.append(data)
        except Exception as e:
            print(f"讀取檔案 {filename} 時發生錯誤: {e}")


In [2]:
result = process_data(data_list)

找不到實體 '無品牌實體' 在上下文中 'RFID喜統入門-無線射頻辨識系統||808U-4 (二手書 七成新)'
找不到實體 '無品牌實體' 在上下文中 '§戀ソ足§告別蝴蝶袖∼120D咖啡因健康美容手臂護套3雙'
找不到實體 'Apple' 在上下文中 '2021 iPad Pro 11吋 512G WiFi 太空灰 (MHQW3TA/A)'
找不到實體 '蘋果' 在上下文中 '2021 iPad Pro 11吋 512G WiFi 太空灰 (MHQW3TA/A)'
找不到實體 'Apple' 在上下文中 'iPhone 12 mini MagSafe 皮革護套 - 粉橘色 Pink Citrus (MHMN3FE/A)'
找不到實體 '無品牌實體' 在上下文中 '2GB  1.8吋魔力搖滾喇叭機'
找不到實體 '松下' 在上下文中 'Panasonic 國際牌 遠紅外線加熱 浴室暖風機 FV-40BF3W 雙馬達 220V 無線遙控'
找不到實體 'AMD' 在上下文中 '(DIY)寒冰火龍(R9-5950X/技嘉X570/32G/1TB M.2/2T/RTX3070)'
找不到實體 'NVIDIA' 在上下文中 '(DIY)寒冰火龍(R9-5950X/技嘉X570/32G/1TB M.2/2T/RTX3070)'
找不到實體 'MARC JACOBS' 在上下文中 'MARC  JACOBS Peanuts x The Snapshot 滿版圖案相機包(墨綠色) H136L01FA21 316'
找不到實體 'Sesame Street' 在上下文中 'Elmo 芝麻街 ❤ 浴帽 (粉)'
找不到實體 '無品牌實體' 在上下文中 '3D 前座防汙椅套 輕薄款'
找不到實體 '無品牌實體' 在上下文中 '[贈品]沙龍套組送蛋糕毛巾'
找不到實體 '優蘭姿' 在上下文中 'Ulanzi VL192智慧藍芽補光燈'
找不到實體 '無品牌實體' 在上下文中 '☆韓貨寬版針織上衣'
找不到實體 '無品牌實體' 在上下文中 '【安卓車機胎外式胎壓偵測器】胎外式 整合型胎壓 android系統 TW601'
找不到實體 '無品牌實體' 在上下文中 '【183 SX】和平鴿印花公主袖T恤(白)'
找不到實體 '無品牌實體' 在上下文中 '【日本進口】日本2020最新! 三層

In [3]:
len(result)

1001

In [4]:
def find_answer(data, context, question):
    for item in data:
        if item['context'] == context and item['question'] == question:
            return item['answer']
    return None  # 如果找不到匹配項，返回 None


gp_file_path = './data/test.pickle'
with open(gp_file_path, 'rb') as file:
    gp_data = pickle.load(file)


y_pred = []
y_true = []

for pred in result:
    context = pred['context']
    question = pred['question']
    answer = pred['answer']
    true_answer = find_answer(gp_data, context, question)

    if true_answer is not None:
        y_pred.append(answer)
        y_true.append(true_answer)
        if answer != true_answer:
            print(f"預測答案: {context}, 屬性: {question}, 答案: {answer}")
            print(f"真實答案: {context}, 屬性: {question}, 答案: {true_answer}")
            print("=" * 150)
    else:
        print(f"找不到匹配的答案: {context} {question}")


print(classification_report(y_true, y_pred, digits = 4, mode='strict'))

預測答案: 《Alice小女人》撞色蕾絲 串珠按摩性感開檔情趣丁字褲-戀人白, 屬性: 品牌, 答案: ['B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
真實答案: 《Alice小女人》撞色蕾絲 串珠按摩性感開檔情趣丁字褲-戀人白, 屬性: 品牌, 答案: ['O', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
預測答案: 《Delser》義大利帝舍嬰兒餅粉400g, 屬性: 品牌, 答案: ['O', 'B', 'I', 'I', 'I', 'I', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
真實答案: 《Delser》義大利帝舍嬰兒餅粉400g, 屬性: 品牌, 答案: ['O', 'B', 'I', 'I', 'I', 'I', 'I', 'O', 'O', 'O', 'O', 'B', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
預測答案: TBL 13328JPB 02 天柏藍 Timberland RADLER 原創潮流時尚錶 , 屬性: 品牌, 答案: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'I', 'O', 'O', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'